# 02-B. Ví dụ ứng dụng — Tối ưu có ràng buộc

Notebook chứa các **bài toán ứng dụng cụ thể** cho tối ưu có ràng
buộc, dùng lại đúng engine (parser, GA tự cài với quy tắc khả thi Deb
+ epsilon giảm dần, SciPy SLSQP) như
`02_A_constrained_optimization.ipynb` — nhưng đặt trực tiếp bằng code
(không qua form nhập liệu) vì đây là các bài mẫu cố định, không cần
người dùng gõ lại mỗi lần.

File này **độc lập, không phụ thuộc** `02_A_constrained_optimization.ipynb`
— toàn bộ engine được sao chép lại ở cell dưới đây, giống cách
`01_unconstrained_optimization.ipynb` / `02_A` / `03_TSP.ipynb` đều tự
chứa hoàn chỉnh.

**Cách dùng:** chạy tất cả các cell (`Run All`). Muốn đổi tham số GA
(quần thể, số thế hệ, số lần chạy) thì sửa trực tiếp trong cell đầu
tiên (`POPULATION_SIZE`, `GENERATIONS`, `N_RUNS`).

In [1]:
import functools
import re
import time

import numpy as np
import scipy
import sympy
sp = sympy

from dataclasses import dataclass
from scipy.optimize import minimize
from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)


# ============================================================
# 1. PARSER
# ============================================================

# LƯU Ý QUAN TRỌNG:
#
# KHÔNG dùng implicit_multiplication_application ở đây, vì nó bao
# gồm split_symbols - transformation này cắt tên biến nhiều ký tự
# thành tích các chữ cái đơn:
#
#       x1^2 + x2^2   ->   x*1**2 + x*2**2   ->   5*x
#       cost          ->   c*o*s*t
#
# tức là chương trình sẽ âm thầm giải một bài toán khác hẳn.
#
# Dùng implicit_multiplication (không có split_symbols) thì
# "2x + 3y" vẫn hiểu được, còn "x1", "x2" giữ nguyên là biến.

TRANSFORMATIONS = standard_transformations + (
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)

# Các hàm / hằng số toán học mà user được phép nhập.
# Đây cũng là các tên KHÔNG được dùng làm tên biến.
LOCAL_DICT = {
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "asin": sp.asin,
    "acos": sp.acos,
    "atan": sp.atan,
    "sinh": sp.sinh,
    "cosh": sp.cosh,
    "tanh": sp.tanh,
    "exp": sp.exp,
    "log": sp.log,
    "ln": sp.log,
    "sqrt": sp.sqrt,
    "abs": sp.Abs,
    "Abs": sp.Abs,
    "pi": sp.pi,
    "e": sp.E,
    "E": sp.E,
}

# Giới hạn namespace mà parse_expr nhìn thấy.
#
# Mặc định parse_expr dùng toàn bộ namespace của sympy, nên những
# tên biến hoàn toàn hợp lệ như N, O, Q, S, I, beta, gamma... bị
# hiểu thành đối tượng sympy thay vì biến (I -> đơn vị ảo,
# N -> hàm evalf, beta -> hàm beta, ...).
#
# Chỉ để lại đúng những gì bộ parse cần để dựng biểu thức; mọi
# tên khác sẽ tự động trở thành Symbol.
GLOBAL_DICT = {
    "Symbol": sp.Symbol,
    "Integer": sp.Integer,
    "Float": sp.Float,
    "Rational": sp.Rational,
}

RESERVED_NAMES = set(LOCAL_DICT)

# Ký hiệu so sánh, xếp để bắt "<=" trước "<"
COMPARISON_PATTERN = re.compile(r"<=|>=|==|<|>|=")


@dataclass
class ParsedConstraint:
    # ineq: expr >= 0
    # eq:   expr = 0
    kind: str
    expr: sp.Expr
    original: str


def parse_math_expr(text):
    """
    Parse biểu thức toán học tự nhiên.

    Ví dụ:
        x^2 + y^2
        2x + 3y
        x1^2 + x2^2        (tên biến nhiều ký tự được giữ nguyên)
        sin(x) + cos(y)
        e^x + log(y)
    """

    text = text.strip()

    if not text:
        raise ValueError("Biểu thức rỗng.")

    try:
        return parse_expr(
            text,
            local_dict=LOCAL_DICT,
            global_dict=GLOBAL_DICT,
            transformations=TRANSFORMATIONS,
            evaluate=True,
        )

    except Exception as error:
        raise ValueError(
            f"Không đọc được biểu thức {text!r}: {error}"
        ) from error


def parse_constraint(text):
    """
    Chuẩn hóa constraint về:

        g(x) >= 0       nếu inequality
        h(x) = 0        nếu equality

    Ví dụ:
        x^2 + y^2 <= 9
    thành:
        9 - x^2 - y^2 >= 0
    """

    text = text.strip()

    operators = COMPARISON_PATTERN.findall(text)

    if not operators:
        raise ValueError(
            f"Ràng buộc {text!r} phải chứa <=, >= hoặc =."
        )

    # Chuỗi kép "0 <= x <= 5" do parse_constraints() tách trước khi
    # gọi vào đây, nên tới đây mà còn nhiều dấu so sánh là thật sự sai.
    if len(operators) > 1:
        raise ValueError(
            f"Ràng buộc {text!r} có quá nhiều dấu so sánh."
        )

    operator = operators[0]

    if operator in ("<", ">"):
        raise ValueError(
            f"Ràng buộc {text!r} dùng bất đẳng thức nghiêm ngặt. "
            "Tối ưu số cần miền đóng, hãy dùng "
            f"'{operator}=' thay cho '{operator}'."
        )

    lhs_text, rhs_text = text.split(operator, 1)

    lhs = parse_math_expr(lhs_text)
    rhs = parse_math_expr(rhs_text)

    if operator == "<=":
        # lhs <= rhs
        # rhs - lhs >= 0
        expr = rhs - lhs
        kind = "ineq"

    elif operator == ">=":
        # lhs >= rhs
        # lhs - rhs >= 0
        expr = lhs - rhs
        kind = "ineq"

    else:
        # lhs = rhs
        # lhs - rhs = 0
        expr = lhs - rhs
        kind = "eq"

    # expand() đủ để gom hạng tử và rẻ hơn simplify() rất nhiều
    return ParsedConstraint(
        kind=kind,
        expr=sp.expand(expr),
        original=text,
    )


def parse_constraints(text):
    """
    Đọc MỘT dòng ràng buộc, trả về danh sách ràng buộc đã chuẩn hóa.

    Bình thường một dòng cho một ràng buộc. Riêng dạng chuỗi kép thì
    tách làm hai:

        0 <= x <= 2     ->     0 <= x     và     x <= 2
    """

    text = text.strip()

    operators = COMPARISON_PATTERN.findall(text)

    if len(operators) <= 1:
        return [parse_constraint(text)]

    if len(operators) > 2:
        raise ValueError(
            f"Ràng buộc {text!r} có nhiều hơn hai dấu so sánh."
        )

    operator = operators[0]

    if operators[1] != operator or operator not in ("<=", ">="):
        raise ValueError(
            f"Ràng buộc {text!r} có hai dấu so sánh không cùng chiều. "
            "Dạng chuỗi kép chỉ nhận 'a <= x <= b' hoặc 'a >= x >= b'."
        )

    # Cắt tại đúng hai vị trí toán tử
    dau = text.index(operator)
    sau = text.index(operator, dau + len(operator))

    trai = text[:dau]
    giua = text[dau + len(operator):sau]
    phai = text[sau + len(operator):]

    return [
        parse_constraint(f"{trai}{operator}{giua}"),
        parse_constraint(f"{giua}{operator}{phai}"),
    ]


# ============================================================
# 2. BUILD OPTIMIZATION PROBLEM
# ============================================================

def _natural_sort_key(symbol):
    """
    Sắp biến theo thứ tự tự nhiên: x1, x2, x10
    thay vì thứ tự chữ cái: x1, x10, x2
    """

    parts = re.split(r"(\d+)", symbol.name)

    return [
        (1, int(part), "") if part.isdigit() else (0, 0, part)
        for part in parts
    ]


def _implicit_products(symbols):
    """
    Suy ra phép nhân ngầm từ tên biến bị dính liền.

    Bỏ split_symbols của sympy là cần thiết để 'x1', 'x2', 'cost' giữ
    nguyên là biến. Cái giá là 'xy' cũng thành một biến, trong khi
    người dùng gõ '2xy' gần như luôn có ý là 2*x*y.

    Quy tắc tách, chỉ dựa vào bằng chứng trong CHÍNH bài toán: một tên
    nhiều chữ cái được tách thành tích khi MỌI chữ cái của nó đều đã
    là biến ở chỗ khác.

        4x^2 - 2xy + 6y^2   ->  có x, có y  ->  xy tách thành x*y
        cost + x            ->  c,o,s,t không phải biến  ->  giữ 'cost'
        x1^2 + x2^2         ->  có chữ số  ->  không đụng tới
        min xy              ->  không có x, y nào khác  ->  giữ 'xy'

    Trả về dict thay thế cho Expr.subs(), rỗng nếu không có gì để tách.
    """

    don_le = {
        symbol.name
        for symbol in symbols
        if len(symbol.name) == 1 and symbol.name.isalpha()
    }

    thay_the = {}

    for symbol in symbols:
        ten = symbol.name

        if len(ten) < 2 or not ten.isalpha():
            continue

        if all(chu in don_le for chu in ten):
            tich = sp.Integer(1)
            for chu in ten:
                tich *= sp.Symbol(chu)
            thay_the[symbol] = tich

    return thay_the


def build_problem(objective_text, constraint_texts):

    # Parse objective
    objective_expr = parse_math_expr(objective_text)

    # Parse constraints (một dòng có thể sinh ra hai ràng buộc)
    constraints = [
        parsed
        for text in constraint_texts
        for parsed in parse_constraints(text)
    ]

    # --------------------------------------------------------
    # Tự động tìm tất cả biến
    # --------------------------------------------------------

    symbols = set(objective_expr.free_symbols)

    for constraint in constraints:
        symbols |= constraint.expr.free_symbols

    # Tách tên dính liền thành phép nhân: '2xy' -> 2*x*y
    thay_the = _implicit_products(symbols)

    if thay_the:
        objective_expr = sp.expand(objective_expr.subs(thay_the))

        constraints = [
            ParsedConstraint(
                kind=c.kind,
                expr=sp.expand(c.expr.subs(thay_the)),
                original=c.original,
            )
            for c in constraints
        ]

        symbols = set(objective_expr.free_symbols)

        for constraint in constraints:
            symbols |= constraint.expr.free_symbols

    variables = sorted(symbols, key=_natural_sort_key)

    if not variables:
        raise ValueError("Không tìm thấy biến quyết định.")

    # --------------------------------------------------------
    # Symbolic -> numerical
    # --------------------------------------------------------

    objective_raw = sp.lambdify(
        variables,
        objective_expr,
        modules="numpy"
    )

    constraint_raw = [
        sp.lambdify(
            variables,
            c.expr,
            modules="numpy"
        )
        for c in constraints
    ]

    # Gradient ky hieu cua tung rang buoc, dung de chuan hoa thang do
    # vi pham (xem build_scaled_constraint_value)
    gradient_raw = [
        sp.lambdify(
            variables,
            [sp.diff(c.expr, v) for v in variables],
            modules="numpy"
        )
        for c in constraints
    ]

    def objective(x):
        try:
            value = float(
                np.asarray(objective_raw(*x)).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.inf

    def constraint_value(index, x):
        try:
            value = float(
                np.asarray(
                    constraint_raw[index](*x)
                ).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.nan

    def constraint_gradient(index, x):
        try:
            gradient = np.asarray(
                gradient_raw[index](*x),
                dtype=float
            ).ravel()

            if np.all(np.isfinite(gradient)):
                return gradient

        except Exception:
            pass

        return np.full(len(variables), np.nan)

    return (
        objective_expr,
        constraints,
        variables,
        objective,
        constraint_value,
        constraint_gradient,
    )


# ============================================================
# 3. CONSTRAINT HANDLING
# ============================================================

# Mức phạt cho điểm nằm ngoài miền xác định (log(âm), chia 0, ...)
OUT_OF_DOMAIN_VIOLATION = 1e6

FEASIBILITY_TOLERANCE = 1e-6

# Hộp dùng để SINH quần thể ban đầu và đặt thang bước đột biến
# (sigma = mutation_scale x độ rộng hộp). KHÔNG phải một cái lồng:
# GA không bị cắt về hộp nên có thể di cư ra ngoài nếu nghiệm nằm
# ngoài. Muốn chặn thật thì viết thành ràng buộc, ví dụ 'x >= 0'.
INIT_BOX = (-5.0, 5.0)

# Tham số GA dùng CHUNG cho cả bản tự cài lẫn PyGAD bên dưới.
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.15
MUTATION_SCALE = 0.08
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3
EPSILON_DECAY_FRACTION = 0.7
EPSILON_DECAY_POWER = 4.0



def build_scaled_constraint_value(
    constraints,
    constraint_value,
    constraint_gradient,
    init_box,
    n_samples=256,
    seed=0,
):
    """
    Chuẩn hóa mỗi ràng buộc theo độ lớn gradient điển hình của nó.

    Vì sao cần: mức vi phạm |h(x)| phụ thuộc vào việc người dùng
    viết ràng buộc thế nào, trong khi tolerance lại là một hằng số
    tuyệt đối. Hai cách viết TƯƠNG ĐƯƠNG cho kết quả khác nhau:

        x + y = 1                  ->  tolerance hiệu dụng 1e-6
        0.001x + 0.001y = 0.001    ->  tolerance hiệu dụng 1e-3

    Cách viết thứ hai từng cho f = 0.49925, tức là THẤP HƠN cực
    tiểu thật 0.5 - một giá trị bất khả thi về mặt toán học, vì
    chương trình coi là khả thi những điểm thực ra còn vi phạm.

    Chia g(x) cho ||∇g|| điển hình làm mức vi phạm trở thành xấp xỉ
    KHOẢNG CÁCH tới mặt ràng buộc: nhân cả ràng buộc với hằng số k
    thì tử số và mẫu số cùng nhân k, kết quả không đổi.

    Dùng một hệ số HẰNG cho mỗi ràng buộc (trung vị của ||∇g|| trên
    một mẫu ngẫu nhiên trong miền) thay vì ||∇g(x)|| tại từng điểm:
    hệ số hằng vẫn đạt bất biến tỉ lệ, đồng thời tránh trường hợp
    gradient suy biến (∇g = 0, ví dụ x^2+y^2-1 tại gốc tọa độ) làm
    mẫu số bằng 0.

    Hệ số dương nên phép chia không đổi dấu, không đổi nghiệm.
    """

    if not constraints:
        return constraint_value

    init_box = np.asarray(init_box, dtype=float)

    rng = np.random.default_rng(seed)

    sample = rng.uniform(
        init_box[:, 0],
        init_box[:, 1],
        size=(n_samples, len(init_box))
    )

    scales = np.ones(len(constraints))

    for i in range(len(constraints)):

        norms = [
            norm
            for norm in (
                float(
                    np.linalg.norm(constraint_gradient(i, x))
                )
                for x in sample
            )
            if np.isfinite(norm) and norm > 0
        ]

        if norms:
            scales[i] = float(np.median(norms))

    def scaled_constraint_value(index, x):
        return constraint_value(index, x) / scales[index]

    scaled_constraint_value.scales = scales

    return scaled_constraint_value


def constraint_violations(x, constraints, constraint_value):
    """
    Mức vi phạm của TỪNG ràng buộc (đơn vị gốc, không bình phương).

    Inequality:
        g(x) >= 0   ->   violation = max(0, -g(x))

    Equality:
        h(x) = 0    ->   violation = |h(x)|
    """

    result = np.empty(len(constraints))

    for i, constraint in enumerate(constraints):

        value = constraint_value(i, x)

        # Không nằm trong miền xác định
        if not np.isfinite(value):
            result[i] = OUT_OF_DOMAIN_VIOLATION
            continue

        if constraint.kind == "ineq":
            result[i] = max(0.0, -value)

        else:
            result[i] = abs(value)

    return result


def total_constraint_violation(x, constraints, constraint_value):
    """
    Tổng mức vi phạm.

    Dùng tổng trị tuyệt đối (không bình phương) để con số báo cáo
    cùng đơn vị với tolerance - bình phương làm vi phạm 1e-6 hiện
    thành 1e-12, trông như đã khả thi trong khi thực ra thì chưa.
    """

    if not constraints:
        return 0.0

    return float(
        constraint_violations(
            x, constraints, constraint_value
        ).sum()
    )


def max_constraint_violation(x, constraints, constraint_value):
    """Vi phạm lớn nhất - đây mới là đại lượng đem so với tolerance."""

    if not constraints:
        return 0.0

    return float(
        constraint_violations(
            x, constraints, constraint_value
        ).max()
    )


def is_feasible(
    x,
    constraints,
    constraint_value,
    tolerance=FEASIBILITY_TOLERANCE
):

    return max_constraint_violation(
        x, constraints, constraint_value
    ) <= tolerance


def is_better(
    objective_a, violation_a,
    objective_b, violation_b,
    tolerance=FEASIBILITY_TOLERANCE
):
    """
    Quy tắc so sánh của Deb (feasibility rules):

        1. Nghiệm khả thi luôn tốt hơn nghiệm bất khả thi.
        2. Hai nghiệm cùng khả thi     -> so f(x).
        3. Hai nghiệm cùng bất khả thi -> so mức vi phạm.

    Trả về True nếu A tốt hơn B.
    """

    feasible_a = violation_a <= tolerance
    feasible_b = violation_b <= tolerance

    if feasible_a != feasible_b:
        return feasible_a

    if feasible_a:
        return objective_a < objective_b

    return violation_a < violation_b


def build_result(
    x,
    objective,
    constraints,
    constraint_value,
    elapsed_time,
    **extra
):
    """Gói kết quả theo một định dạng chung cho mọi phương pháp."""

    x = np.asarray(x, dtype=float)

    result = {
        "x": x,

        "fun": objective(x),

        "feasible": is_feasible(
            x, constraints, constraint_value
        ),

        "violation": total_constraint_violation(
            x, constraints, constraint_value
        ),

        "max_violation": max_constraint_violation(
            x, constraints, constraint_value
        ),

        "time": elapsed_time,
    }

    result.update(extra)

    return result


# ============================================================
# 4. GENETIC ALGORITHM
# ============================================================

def genetic_algorithm(
    objective,
    constraints,
    constraint_value,
    init_box,

    population_size=100,
    generations=500,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    mutation_scale=MUTATION_SCALE,

    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,

    feasibility_tolerance=FEASIBILITY_TOLERANCE,
    epsilon_decay_fraction=EPSILON_DECAY_FRACTION,
    epsilon_decay_power=EPSILON_DECAY_POWER,

    reject_infeasible=True,
    rejection_min_share=0.2,

    seed=42,
):
    """
    GA mã hóa số thực cho bài toán tối ưu có ràng buộc.

    Xử lý ràng buộc bằng quy tắc khả thi của Deb kết hợp ngưỡng
    epsilon giảm dần (Takahama & Sato), KHÔNG dùng hệ số phạt tĩnh.

    Vì sao bỏ penalty tĩnh: với fitness = f + 1e6 * violation,
    thang phạt át hoàn toàn hàm mục tiêu, nên quần thể chỉ tối
    thiểu vi phạm rồi đứng yên tại một điểm bất kỳ trên mặt ràng
    buộc. Đo trên 'min x^2+y^2 s.t. x+y=1' (nghiệm đúng 0.5),
    penalty 1e6 cho trung bình 2.71 và xấu nhất 6.27 qua 8 seed.

    Ngưỡng epsilon nới lỏng ràng buộc ở giai đoạn đầu để quần thể
    còn di chuyển được dọc theo mặt ràng buộc - điều thiết yếu với
    ràng buộc đẳng thức, nơi tập khả thi có độ đo bằng 0 - rồi
    siết dần về feasibility_tolerance.
    """

    rng = np.random.default_rng(seed)

    init_box = np.asarray(init_box, dtype=float)

    lower = init_box[:, 0]
    upper = init_box[:, 1]

    variable_range = upper - lower

    n_variables = len(init_box)

    # --------------------------------------------------------
    # Initial population
    # --------------------------------------------------------

    population = rng.uniform(
        lower,
        upper,
        size=(population_size, n_variables)
    )

    # --------------------------------------------------------
    # Đánh giá: tách riêng mục tiêu và mức vi phạm
    # --------------------------------------------------------

    def evaluate(pop):

        objectives = np.empty(len(pop))

        # Vi phạm của TỪNG ràng buộc, cần cho việc loại cá thể bất khả thi
        tung_rang_buoc = np.zeros((len(pop), max(1, len(constraints))))

        for i, individual in enumerate(pop):

            objectives[i] = objective(individual)

            if constraints:
                tung_rang_buoc[i] = constraint_violations(
                    individual,
                    constraints,
                    constraint_value
                )

        violations = tung_rang_buoc.sum(axis=1) if constraints \
            else np.zeros(len(pop))

        # Điểm ngoài miền xác định: giữ hữu hạn để còn sắp xếp được
        objectives = np.where(
            np.isfinite(objectives),
            objectives,
            np.finfo(float).max
        )

        return objectives, violations, tung_rang_buoc

    # --------------------------------------------------------
    # Xếp hạng theo quy tắc Deb với ngưỡng epsilon
    # --------------------------------------------------------

    def rank_order(objectives, violations, epsilon):

        infeasible = violations > epsilon

        secondary = np.where(
            infeasible,
            violations,
            objectives
        )

        # lexsort: khóa cuối cùng là khóa chính
        return np.lexsort(
            (secondary, infeasible.astype(np.int64))
        )

    # --------------------------------------------------------
    # Lịch giảm epsilon
    # --------------------------------------------------------

    cutoff = max(
        1,
        int(epsilon_decay_fraction * generations)
    )

    def epsilon_at(generation, epsilon_0):

        if generation >= cutoff:
            return feasibility_tolerance

        factor = (
            1.0 - generation / cutoff
        ) ** epsilon_decay_power

        return max(
            feasibility_tolerance,
            epsilon_0 * factor
        )

    # --------------------------------------------------------
    # Tournament selection (theo thứ hạng Deb)
    # --------------------------------------------------------

    def tournament_selection(rank, be_lai_tao):

        indices = be_lai_tao[
            rng.integers(0, len(be_lai_tao), size=tournament_size)
        ]

        best_index = indices[
            np.argmin(rank[indices])
        ]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Loại cá thể bất khả thi khỏi bể lai tạo
    # --------------------------------------------------------

    def be_lai_tao_cua(tung_rang_buoc, epsilon):
        """
        Cá thể vi phạm ràng buộc thì không được làm cha mẹ.

        Chỉ áp cho những ràng buộc mà một phần đủ lớn của quần thể
        (rejection_min_share) đang thỏa mãn. Lý do: tập khả thi của
        ràng buộc ĐẲNG THỨC có độ đo bằng 0, gần như không cá thể nào
        thỏa ở những thế hệ đầu - loại thẳng thì cả quần thể chết và
        thuật toán không khởi động được. Ràng buộc kiểu 'r >= 0.1' thì
        luôn có sẵn nhiều cá thể thỏa, nên lọc được ngay từ đầu.

        Dùng đúng epsilon hiện tại (không phải feasibility_tolerance cố
        định) để xét "thỏa mãn": epsilon giảm dần qua các thế hệ nên
        vùng "được phép sinh sản" cũng SIẾT DẦN theo đúng lịch epsilon_at()
        - nếu dùng feasibility_tolerance cứng thì với ràng buộc đẳng thức,
        hầu như không cá thể nào lọt qua cho tới tận cuối, khiến cơ chế
        này gần như không có tác dụng suốt phần lớn quá trình tiến hóa.
        """

        tat_ca = np.arange(len(tung_rang_buoc))

        if not constraints or not reject_infeasible:
            return tat_ca

        thoa = tung_rang_buoc <= epsilon

        ap_dung = thoa.mean(axis=0) >= rejection_min_share

        if not ap_dung.any():
            return tat_ca

        giu = thoa[:, ap_dung].all(axis=1)

        # Còn quá ít cá thể thì không đủ đa dạng để lai tạo
        if giu.sum() < max(2 * elite_size, tournament_size):
            return tat_ca

        return tat_ca[giu]

    # --------------------------------------------------------
    # Blend crossover
    # --------------------------------------------------------

    def crossover(parent1, parent2):

        if rng.random() > crossover_rate:
            return (
                parent1.copy(),
                parent2.copy()
            )

        alpha = rng.uniform(
            -0.25,
            1.25,
            size=n_variables
        )

        child1 = (
            alpha * parent1
            + (1 - alpha) * parent2
        )

        child2 = (
            alpha * parent2
            + (1 - alpha) * parent1
        )

        # KHÔNG cắt về hộp: hộp chỉ dùng để khởi tạo và đặt thang
        # bước đột biến, không phải một cái lồng. Áp lực chọn lọc tự
        # kéo quần thể tới vùng tốt, kể cả khi vùng đó nằm ngoài hộp.
        # Đo trên 'min (x-10)^2+(y-10)^2' với hộp [-5,5]: có cắt thì
        # kẹt ở f=50 tại (5,5), bỏ cắt thì ra đúng f=0 tại (10,10).
        return child1, child2

    # --------------------------------------------------------
    # Gaussian mutation
    # --------------------------------------------------------

    def mutate(child):

        mutation_mask = (
            rng.random(n_variables)
            < mutation_rate
        )

        if np.any(mutation_mask):

            child[mutation_mask] += rng.normal(
                loc=0,
                scale=(
                    mutation_scale
                    * variable_range[mutation_mask]
                )
            )

        return child

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []
    history_violation = []

    start_time = time.perf_counter()

    objectives, violations, tung_rang_buoc = evaluate(population)

    # Ngưỡng epsilon ban đầu: vi phạm trung vị của quần thể đầu tiên
    epsilon_0 = float(np.median(violations))

    best_index = rank_order(
        objectives, violations, feasibility_tolerance
    )[0]

    best_solution = population[best_index].copy()
    best_objective = objectives[best_index]
    best_violation = violations[best_index]

    def update_best():

        nonlocal best_solution, best_objective, best_violation

        for i in range(len(population)):
            if is_better(
                objectives[i], violations[i],
                best_objective, best_violation,
                feasibility_tolerance
            ):
                best_solution = population[i].copy()
                best_objective = objectives[i]
                best_violation = violations[i]

    for generation in range(generations):

        epsilon = epsilon_at(generation, epsilon_0)

        order = rank_order(objectives, violations, epsilon)

        rank = np.empty(population_size, dtype=np.int64)
        rank[order] = np.arange(population_size)

        be_lai_tao = be_lai_tao_cua(tung_rang_buoc, epsilon)

        # Nghiệm tốt nhất từng gặp, xét theo tolerance thật
        update_best()

        # best_objective có thể là sentinel np.finfo(float).max khi cá
        # thể ít vi phạm nhất lại có f(x) không tính được (NaN, ví dụ
        # ln(x) với x<=0) - lúc đó chỉ số vi phạm mới có nghĩa, giá trị
        # f(x) sentinel không đại diện cho gì cả nên ghi NaN để vẽ đồ
        # thị bỏ qua, thay vì làm trục giãn ra tới 1e+308.
        history.append(
            np.nan if best_objective == np.finfo(float).max else best_objective
        )
        history_violation.append(best_violation)

        # Elitism
        new_population = [
            population[i].copy()
            for i in order[:elite_size]
        ]

        # Sinh thế hệ tiếp theo
        while len(new_population) < population_size:

            parent1 = tournament_selection(rank, be_lai_tao)
            parent2 = tournament_selection(rank, be_lai_tao)

            child1, child2 = crossover(
                parent1,
                parent2
            )

            new_population.append(
                mutate(child1)
            )

            if len(new_population) < population_size:
                new_population.append(
                    mutate(child2)
                )

        population = np.asarray(new_population)

        objectives, violations, tung_rang_buoc = evaluate(population)

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    update_best()

    elapsed_time = time.perf_counter() - start_time

    # Thế hệ đầu tiên mà nghiệm tốt nhất từng gặp đã khả thi.
    # None nghĩa là chạy hết số thế hệ vẫn chưa thỏa mãn ràng buộc.
    feasible_at = next(
        (
            g
            for g, violation in enumerate(history_violation)
            if violation <= feasibility_tolerance
        ),
        None,
    )

    # Thế hệ sớm nhất mà cặp (objective, violation) tốt nhất cuối cùng
    # đã đạt được — chỉ là chỉ số báo cáo, KHÔNG dừng vòng lặp sớm.
    generations_run = next(
        (
            g + 1
            for g, (obj_g, vio_g) in enumerate(zip(history, history_violation))
            if obj_g == best_objective and vio_g == best_violation
        ),
        generations,
    )

    return build_result(
        best_solution,
        objective,
        constraints,
        constraint_value,
        elapsed_time,
        history=history,
        history_violation=history_violation,
        generations=generations,
        generations_run=generations_run,
        feasible_at=feasible_at,
        seed=seed,
    )


# ============================================================
# 5. SCIPY - SLSQP, DÙNG LÀM MỐC SO SÁNH
# ============================================================

def _run_slsqp(
    objective,
    constraints,
    constraint_value,
    x0
):
    """Một lần chạy SLSQP từ điểm khởi tạo x0."""

    scipy_constraints = [
        {
            # SLSQP: "ineq" nghĩa là g(x) >= 0, "eq" nghĩa là
            # h(x) = 0 - trùng đúng dạng đã chuẩn hóa ở
            # parse_constraint, nên dùng thẳng constraint.kind
            "type": constraint.kind,

            "fun":
                lambda x, i=i:
                constraint_value(i, x)
        }
        for i, constraint in enumerate(constraints)
    ]

    return minimize(
        objective,

        x0=x0,

        method="SLSQP",

        constraints=scipy_constraints,

        options={
            "maxiter": 2000,
            "ftol": 1e-12,
            "disp": False,
        }
    )


def scipy_slsqp(
    objective,
    constraints,
    constraint_value,
    init_box,
    n_starts=30,
    seed=42,
):
    """
    SLSQP đa điểm khởi tạo (multi-start).

    Vì sao cần nhiều điểm: SLSQP là thuật toán CỤC BỘ, nó hội tụ về
    điểm dừng KKT gần nhất chứ không phải cực tiểu toàn cục. Với
    'min x*y s.t. x^2+y^2=1' (nghiệm đúng -0.5), khởi tạo từ trung
    điểm init_box (0,0) cho ra +0.5 - tức là điểm CỰC ĐẠI.

    SLSQP KHÔNG nhận init_box: chặn trên/dưới nếu cần thì viết thành
    ràng buộc (ví dụ 'x >= 0'), để GA và SLSQP giải đúng cùng một
    bài toán. init_box chỉ dùng để rải điểm khởi tạo.
    """

    init_box = [tuple(b) for b in init_box]

    rng = np.random.default_rng(seed)

    lower = np.array([b[0] for b in init_box], dtype=float)
    upper = np.array([b[1] for b in init_box], dtype=float)

    start_points = [(lower + upper) / 2]

    if n_starts > 1:
        start_points.extend(
            rng.uniform(
                lower, upper,
                size=(n_starts - 1, len(init_box))
            )
        )

    start_time = time.perf_counter()

    best = None
    n_success = 0

    for x0 in start_points:

        try:
            raw = _run_slsqp(
                objective,
                constraints,
                constraint_value,
                x0
            )

        except Exception:
            continue

        x = np.asarray(raw.x, dtype=float)

        value = objective(x)

        if not np.isfinite(value):
            continue

        violation = total_constraint_violation(
            x, constraints, constraint_value
        )

        n_success += bool(raw.success)

        if best is None or is_better(
            value, violation,
            best[1], best[2]
        ):
            best = (x, value, violation, raw)

    elapsed_time = time.perf_counter() - start_time

    if best is None:
        raise RuntimeError(
            "SLSQP không tìm được nghiệm hữu hạn từ bất kỳ "
            "điểm khởi tạo nào."
        )

    x, _, _, raw = best

    return build_result(
        x,
        objective,
        constraints,
        constraint_value,
        elapsed_time,
        success=bool(raw.success),
        message=str(raw.message),
        iterations=int(raw.nit),
        generations_run=int(raw.nit),
        n_starts=len(start_points),
        n_success=n_success,
    )


print(f"numpy {np.__version__} | scipy {scipy.__version__} | sympy {sympy.__version__}")
print("Tên dành riêng (không dùng làm biến):", ", ".join(sorted(RESERVED_NAMES)))


# ------------------------------------------------------------------
# Hiển thị dạng ký hiệu toán học (LaTeX)
# ------------------------------------------------------------------
from IPython.display import Markdown, display


def _num(value, digits=10):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"


def show_problem(objective_expr, constraints, variables):
    """Phát biểu bài toán tối ưu dưới dạng toán học chuẩn."""
    bien = ", ".join(sympy.latex(v) for v in variables)
    dong = [r"&\underset{" + bien + r"}{\text{minimize}} \quad && f\left("
            + bien + r"\right) = " + sympy.latex(objective_expr) + r" \\"]

    dau = True
    for c in constraints:
        quan_he = r"\ \ge\ 0" if c.kind == "ineq" else r"\ =\ 0"
        nhan = r"\text{subject to}" if dau else ""
        dong.append("&" + nhan + r" \quad && " + sympy.latex(c.expr) + quan_he + r" \\")
        dau = False

    display(Markdown("$$\n\\begin{aligned}\n" + "\n".join(dong) + "\n\\end{aligned}\n$$"))


def show_solution(name, result, variables):
    """Nghiệm tìm được và thời gian chạy."""
    toado = r" \\ ".join(
        sympy.latex(v) + " &= " + _num(x) for v, x in zip(variables, result["x"])
    )

    khoi = [
        "**" + name + "**",
        "",
        r"$$\begin{aligned}" + toado + r"\end{aligned}$$",
        r"$$f^{*} = " + _num(result["fun"]) + r"$$",
        "",
        "| | |",
        "|---|---|",
        "| Thời gian chạy | $" + _num(result["time"], 6) + r"\ \text{s}$ |",
    ]

    if "feasible_at" in result:
        if result["feasible_at"] is None:
            khoi.append("| Thỏa mãn ràng buộc | chưa đạt sau $"
                        + str(result["generations"]) + "$ thế hệ |")
        else:
            khoi.append("| Thỏa mãn ràng buộc từ thế hệ | $"
                        + str(result["feasible_at"]) + " / "
                        + str(result["generations"]) + "$ |")

    if "success" in result:
        khoi.append("| SLSQP hội tụ | $" + str(result["n_success"]) + "/"
                    + str(result["n_starts"]) + "$ điểm khởi tạo |")

    # Chỉ hiện khi nghiệm KHÔNG khả thi. Lúc bình thường không có dòng
    # này; nhưng nếu thuật toán thất bại thì phải báo, không thì người
    # dùng đọc một con số vô nghĩa mà tưởng là kết quả.
    if not result["feasible"]:
        khoi += [
            "",
            "> ⚠️ **Nghiệm này KHÔNG thỏa mãn ràng buộc** — vi phạm $"
            + _num(result["max_violation"])
            + r"$, vượt dung sai $10^{-6}$. Giá trị $f^{*}$ ở trên không dùng được.",
        ]

    display(Markdown("\n".join(khoi)))


def show_statistics(runs):
    """Thống kê qua nhiều lần chạy GA độc lập."""
    gia_tri = np.array([r["fun"] for r in runs if np.isfinite(r["fun"])])

    if len(gia_tri) == 0:
        display(Markdown("*Không lần chạy nào cho giá trị hữu hạn.*"))
        return

    display(Markdown("\n".join([
        "**Thống kê qua " + str(len(runs)) + " lần chạy độc lập**",
        "",
        "| | |",
        "|---|---|",
        r"| Tốt nhất | $\min f = " + _num(gia_tri.min()) + "$ |",
        r"| Trung bình | $\bar{f} = " + _num(gia_tri.mean()) + "$ |",
        r"| Tệ nhất | $\max f = " + _num(gia_tri.max()) + "$ |",
        r"| Độ lệch chuẩn | $\sigma = " + _num(gia_tri.std()) + "$ |",
    ])))


def show_comparison(results, variables):
    """Bảng so sánh nhiều phương pháp: f*, tọa độ nghiệm.

    Không so thời gian chạy / số thế hệ-vòng lặp: hai thuật toán khác
    bản chất (GA dựa trên quần thể, SLSQP cục bộ theo gradient) nên hai
    con số đó không cùng đơn vị so sánh, dễ gây hiểu lầm.

    `results` là danh sách [(tên, result), ...]."""
    cot_bien = " | ".join("$" + sympy.latex(v) + "$" for v in variables)

    dong = [
        r"| Phương pháp | $f^{*}$ | " + cot_bien + " |",
        "|---|---|" + "---|" * len(variables),
    ]

    for ten, r in results:
        toado = " | ".join("$" + _num(x) + "$" for x in r["x"])
        dong.append("| " + ten + " | $" + _num(r["fun"]) + "$ | " + toado + " |")

    display(Markdown("\n".join(dong)))


# ------------------------------------------------------------------
# Tham số GA/SLSQP dùng cho các bài mẫu dưới đây — không có form nhập
# liệu như 02_A nên đặt cố định ở đây, cùng giá trị mặc định với
# 02_A_constrained_optimization.ipynb để dễ đối chiếu.
# ------------------------------------------------------------------
POPULATION_SIZE = 1000
GENERATIONS = 500
N_RUNS = 3
SLSQP_N_STARTS = 30


numpy 1.26.4 | scipy 1.17.1 | sympy 1.14.0
Tên dành riêng (không dùng làm biến): Abs, E, abs, acos, asin, atan, cos, cosh, e, exp, ln, log, pi, sin, sinh, sqrt, tan, tanh


---
## ① Entropy tối đa (Maximum Entropy)

Bài toán kinh điển (Jaynes — "con xúc xắc bị làm lệch"): tìm phân phối
xác suất $x_1,\dots,x_n$ có **entropy lớn nhất** (ít giả định nhất có
thể, theo nguyên lý cực đại entropy) khi chỉ biết trước một ràng buộc
về kỳ vọng:

$$\max_{x} H(x) = -\sum_{i=1}^{n} x_i \ln x_i \quad
\text{s.t.} \quad \sum_{i=1}^{n} x_i = 1,\ \
\sum_{i=1}^{n} a_i x_i = b,\ \ x_i \ge 0$$

GA và SLSQP ở đây đều tối **thiểu** hóa, nên bài toán được đổi dấu
thành $\min f(x) = \sum_i x_i \ln x_i = -H(x)$ — tương đương hoàn
toàn, không cần sửa gì trong thuật toán.

**Bài mẫu**: xúc xắc 6 mặt ($n=6$, $a_i = i$), biết trước kỳ vọng
$E[X] = b = 4.5$ (xúc xắc công bằng có $E[X]=3.5$, nên đây là một con
xúc xắc bị lệch nhẹ về các mặt lớn). Theo điều kiện KKT/Lagrange,
nghiệm có dạng cấp số nhân $x_i^{*} = C \cdot r^{i}$; giải số phương
trình ràng buộc kỳ vọng cho $r \approx 1.4492539954$, từ đó:

$$x^{*} \approx (0.054353,\ 0.078772,\ 0.114160,\ 0.165447,\
0.239774,\ 0.347494), \qquad H(x^{*}) \approx 1.6135810982$$

> **Lưu ý**: đây là bài toán khó hơn hẳn bài mặc định ở
> `02_A_constrained_optimization.ipynb` (2 ràng buộc đẳng thức đồng
> thời + 6 ràng buộc $x_i\ge0$, thay vì chỉ 1 ràng buộc). Với Quần thể
> 1000 / Số thế hệ 500, GA tự cài đạt khả thi sau khoảng 330-340 thế
> hệ (tùy lần chạy) — chậm hơn hẳn bài mặc định vì phải "trượt" chính
> xác dọc giao tuyến của hai mặt ràng buộc đẳng thức cùng lúc. SLSQP
> giải theo điều kiện KKT nên hội tụ gần như ngay lập tức, không gặp
> vấn đề này.

In [2]:
# ------------------------------------------------------------------
# Bài toán Entropy tối đa — "con xúc xắc bị làm lệch" (Jaynes)
# ------------------------------------------------------------------
#
#   max  H(x) = -sum_i x_i * ln(x_i)
#   s.t. sum_i x_i = 1
#        sum_i a_i * x_i = b        (a_i, b cho trước)
#        x_i >= 0,  i = 1..n
#
# GA/SLSQP ở đây đều tối THIỂU hóa, nên minimize f(x) = sum x_i ln(x_i)
# (= -H(x)) tương đương với tối đa hóa entropy — không cần sửa gì
# trong genetic_algorithm()/scipy_slsqp(), chỉ đổi dấu bài toán khi
# phát biểu bằng chữ.

n = 6
a = list(range(1, n + 1))
b = 4.5

entropy_objective_text = " + ".join(f"x{i}*ln(x{i})" for i in range(1, n + 1))
entropy_constraint_texts = (
    [" + ".join(f"x{i}" for i in range(1, n + 1)) + " = 1"]
    + [" + ".join(f"{a[i - 1]}*x{i}" for i in range(1, n + 1)) + f" = {b}"]
    + [f"x{i} >= 0" for i in range(1, n + 1)]
)

(entropy_objective_expr, entropy_constraints, entropy_variables,
 entropy_objective, entropy_constraint_value,
 entropy_constraint_gradient) = build_problem(
    entropy_objective_text, entropy_constraint_texts
)

# Biến là xác suất trong (0, 1] — khởi tạo/thang đột biến riêng cho
# bài toán này thay vì dùng INIT_BOX mặc định (-5, 5).
entropy_init_box = [(1e-3, 1.0)] * n

entropy_constraint_value = build_scaled_constraint_value(
    entropy_constraints, entropy_constraint_value,
    entropy_constraint_gradient, entropy_init_box,
)

show_problem(entropy_objective_expr, entropy_constraints, entropy_variables)

entropy_ga_runs = [
    genetic_algorithm(
        entropy_objective, entropy_constraints, entropy_constraint_value,
        entropy_init_box,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]
entropy_ga_result = functools.reduce(
    lambda x, y: y if is_better(
        y["fun"], y["violation"], x["fun"], x["violation"]
    ) else x,
    entropy_ga_runs,
)

entropy_scipy_result = scipy_slsqp(
    entropy_objective, entropy_constraints, entropy_constraint_value,
    entropy_init_box,
    n_starts=SLSQP_N_STARTS,
    seed=42,
)

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    entropy_ga_result, entropy_variables,
)

if N_RUNS > 1:
    show_statistics(entropy_ga_runs)

show_solution(
    "SCIPY SLSQP — đa điểm khởi tạo",
    entropy_scipy_result, entropy_variables,
)

show_comparison(
    [
        ("Genetic Algorithm", entropy_ga_result),
        ("SciPy SLSQP", entropy_scipy_result),
    ],
    entropy_variables,
)


$$
\begin{aligned}
&\underset{x_{1}, x_{2}, x_{3}, x_{4}, x_{5}, x_{6}}{\text{minimize}} \quad && f\left(x_{1}, x_{2}, x_{3}, x_{4}, x_{5}, x_{6}\right) = x_{1} \log{\left(x_{1} \right)} + x_{2} \log{\left(x_{2} \right)} + x_{3} \log{\left(x_{3} \right)} + x_{4} \log{\left(x_{4} \right)} + x_{5} \log{\left(x_{5} \right)} + x_{6} \log{\left(x_{6} \right)} \\
&\text{subject to} \quad && x_{1} + x_{2} + x_{3} + x_{4} + x_{5} + x_{6} - 1\ =\ 0 \\
& \quad && x_{1} + 2 x_{2} + 3 x_{3} + 4 x_{4} + 5 x_{5} + 6 x_{6} - 4.5\ =\ 0 \\
& \quad && x_{1}\ \ge\ 0 \\
& \quad && x_{2}\ \ge\ 0 \\
& \quad && x_{3}\ \ge\ 0 \\
& \quad && x_{4}\ \ge\ 0 \\
& \quad && x_{5}\ \ge\ 0 \\
& \quad && x_{6}\ \ge\ 0 \\
\end{aligned}
$$

<lambdifygenerated-1>:2: RuntimeWarning: invalid value encountered in log
  return x1*log(x1) + x2*log(x2) + x3*log(x3) + x4*log(x4) + x5*log(x5) + x6*log(x6)


<lambdifygenerated-1>:2: RuntimeWarning: divide by zero encountered in log
  return x1*log(x1) + x2*log(x2) + x3*log(x3) + x4*log(x4) + x5*log(x5) + x6*log(x6)
<lambdifygenerated-1>:2: RuntimeWarning: invalid value encountered in scalar multiply
  return x1*log(x1) + x2*log(x2) + x3*log(x3) + x4*log(x4) + x5*log(x5) + x6*log(x6)


**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x_{1} &= 0.0376183833 \\ x_{2} &= 0.0654623329 \\ x_{3} &= 0.1210433197 \\ x_{4} &= 0.2089568364 \\ x_{5} &= 0.2690197831 \\ x_{6} &= 0.2979005435\end{aligned}$$
$$f^{*} = -1.5985825694$$

| | |
|---|---|
| Thời gian chạy | $112.513838\ \text{s}$ |
| Thỏa mãn ràng buộc từ thế hệ | $335 / 500$ |

**Thống kê qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = -1.5985825694$ |
| Trung bình | $\bar{f} = -1.5982115195$ |
| Tệ nhất | $\max f = -1.5975995197$ |
| Độ lệch chuẩn | $\sigma = 0.0004359964$ |

**SCIPY SLSQP — đa điểm khởi tạo**

$$\begin{aligned}x_{1} &= 0.0543531685 \\ x_{2} &= 0.0787715488 \\ x_{3} &= 0.1141599779 \\ x_{4} &= 0.1654467928 \\ x_{5} &= 0.2397744429 \\ x_{6} &= 0.3474940691\end{aligned}$$
$$f^{*} = -1.6135810982$$

| | |
|---|---|
| Thời gian chạy | $3.487071\ \text{s}$ |
| SLSQP hội tụ | $30/30$ điểm khởi tạo |

| Phương pháp | $f^{*}$ | $x_{1}$ | $x_{2}$ | $x_{3}$ | $x_{4}$ | $x_{5}$ | $x_{6}$ |
|---|---|---|---|---|---|---|---|
| Genetic Algorithm | $-1.5985825694$ | $0.0376183833$ | $0.0654623329$ | $0.1210433197$ | $0.2089568364$ | $0.2690197831$ | $0.2979005435$ |
| SciPy SLSQP | $-1.6135810982$ | $0.0543531685$ | $0.0787715488$ | $0.1141599779$ | $0.1654467928$ | $0.2397744429$ | $0.3474940691$ |

---
## ② Điểm gần nhất trong miền ràng buộc (projection onto convex set)

Cho một điểm $(x_0,y_0,z_0)$ **nằm ngoài miền ràng buộc**, tìm điểm
$(x,y,z)$ **trong miền ràng buộc** gần nó nhất theo khoảng cách Euclid
bình phương — bài toán chiếu điểm lên một tập lồi:

$$\min f(x,y,z) = (x-x_0)^2 + (y-y_0)^2 + (z-z_0)^2 \quad \text{s.t.}
\quad x+2y+z=4,\ \ x^2+y^2\le9,\ \ z\ge0$$

Miền ràng buộc (giao của một mặt phẳng, một hình trụ, và một nửa
không gian) là **tập lồi**, hàm mục tiêu cũng lồi, nên bài toán có
đúng **một** cực tiểu toàn cục — không có cực trị cục bộ nào khác để
GA/SLSQP bị nhầm.

**Bài mẫu**: $(x_0,y_0,z_0) = (5,\ 4,\ -2)$ — nằm ngoài cả hình trụ
lẫn nửa không gian $z\ge0$. Giải bằng SLSQP đa điểm khởi tạo cho
nghiệm đúng:

$$x^{*}\approx2.9540659199,\quad y^{*}\approx0.5229670393,\quad
z^{*}\approx0, \qquad f^{*}\approx20.2756044629$$

Cả **hai** ràng buộc bất đẳng thức đều **chặt** (active) tại nghiệm —
$x^{*2}+y^{*2}=9$ đúng bằng biên hình trụ, và $z^{*}=0$ đúng bằng biên
$z\ge0$ — một ví dụ tốt về điều kiện KKT khi nhiều ràng buộc cùng chặt
một lúc.

In [3]:
# ------------------------------------------------------------------
# Điểm gần nhất trong miền ràng buộc — chiếu điểm (x0,y0,z0) lên tập lồi
# {x+2y+z=4} ∩ {x^2+y^2<=9} ∩ {z>=0}
# ------------------------------------------------------------------

nearest_x0, nearest_y0, nearest_z0 = 5.0, 4.0, -2.0

nearest_objective_text = (
    f"(x-{nearest_x0})^2 + (y-{nearest_y0})^2 + (z-({nearest_z0}))^2"
)
nearest_constraint_texts = [
    "x + 2*y + z = 4",
    "x^2 + y^2 <= 9",
    "z >= 0",
]

(nearest_objective_expr, nearest_constraints, nearest_variables,
 nearest_objective, nearest_constraint_value,
 nearest_constraint_gradient) = build_problem(
    nearest_objective_text, nearest_constraint_texts
)

nearest_init_box = [INIT_BOX] * len(nearest_variables)

nearest_constraint_value = build_scaled_constraint_value(
    nearest_constraints, nearest_constraint_value,
    nearest_constraint_gradient, nearest_init_box,
)

show_problem(nearest_objective_expr, nearest_constraints, nearest_variables)

nearest_ga_runs = [
    genetic_algorithm(
        nearest_objective, nearest_constraints, nearest_constraint_value,
        nearest_init_box,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]
nearest_ga_result = functools.reduce(
    lambda x, y: y if is_better(
        y["fun"], y["violation"], x["fun"], x["violation"]
    ) else x,
    nearest_ga_runs,
)

nearest_scipy_result = scipy_slsqp(
    nearest_objective, nearest_constraints, nearest_constraint_value,
    nearest_init_box,
    n_starts=SLSQP_N_STARTS,
    seed=42,
)

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    nearest_ga_result, nearest_variables,
)

if N_RUNS > 1:
    show_statistics(nearest_ga_runs)

show_solution(
    "SCIPY SLSQP — đa điểm khởi tạo",
    nearest_scipy_result, nearest_variables,
)

show_comparison(
    [
        ("Genetic Algorithm", nearest_ga_result),
        ("SciPy SLSQP", nearest_scipy_result),
    ],
    nearest_variables,
)


$$
\begin{aligned}
&\underset{x, y, z}{\text{minimize}} \quad && f\left(x, y, z\right) = \left(x - 5.0\right)^{2} + \left(y - 4.0\right)^{2} + \left(z + 2.0\right)^{2} \\
&\text{subject to} \quad && x + 2 y + z - 4\ =\ 0 \\
& \quad && - x^{2} - y^{2} + 9\ \ge\ 0 \\
& \quad && z\ \ge\ 0 \\
\end{aligned}
$$

**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x &= 2.8538164792 \\ y &= 0.5730928965 \\ z &= -4.1748 \times 10^{-8}\end{aligned}$$
$$f^{*} = 20.3497958340$$

| | |
|---|---|
| Thời gian chạy | $86.276193\ \text{s}$ |
| Thỏa mãn ràng buộc từ thế hệ | $331 / 500$ |

**Thống kê qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = 20.3497958340$ |
| Trung bình | $\bar{f} = 20.4814922021$ |
| Tệ nhất | $\max f = 20.5988086766$ |
| Độ lệch chuẩn | $\sigma = 0.1021663185$ |

**SCIPY SLSQP — đa điểm khởi tạo**

$$\begin{aligned}x &= 2.9540659682 \\ y &= 0.5229670159 \\ z &= -3.2842 \times 10^{-14}\end{aligned}$$
$$f^{*} = 20.2756044350$$

| | |
|---|---|
| Thời gian chạy | $0.621633\ \text{s}$ |
| SLSQP hội tụ | $26/30$ điểm khởi tạo |

| Phương pháp | $f^{*}$ | $x$ | $y$ | $z$ |
|---|---|---|---|---|
| Genetic Algorithm | $20.3497958340$ | $2.8538164792$ | $0.5730928965$ | $-4.1748 \times 10^{-8}$ |
| SciPy SLSQP | $20.2756044350$ | $2.9540659682$ | $0.5229670159$ | $-3.2842 \times 10^{-14}$ |

---
## ③ Quy hoạch tuyến tính (Linear Programming — hỗn hợp)

$$\max_{x,y,z} f(x,y,z) = 3x+5y-2z \quad \text{s.t.} \quad
x+2y+z=10,\ \ 2x+y\le8,\ \ x,y,z\ge0$$

Đổi dấu để tối thiểu hóa: $\min -f = -3x-5y+2z$. Mọi thành phần đều
**tuyến tính** (hàm mục tiêu lẫn ràng buộc) — đây là quy hoạch tuyến
tính (LP) thuần túy, nghiệm tối ưu luôn nằm ở một **đỉnh** của miền đa
diện lồi (feasible polytope).

Vì là LP, có thể giải bằng thuật toán chuyên dụng
(`scipy.optimize.linprog`, phương pháp simplex/interior-point) cho
nghiệm **chính xác tuyệt đối** không cần multi-start:

$$x^{*}=2,\quad y^{*}=4,\quad z^{*}=0, \qquad f^{*}_{\max}=26\ \
\left(\min(-f)=-26\right)$$

Ràng buộc $2x+y\le8$ **chặt** ($2(2)+4=8$ đúng bằng biên); $z^{*}=0$
cũng chặt.

In [4]:
# ------------------------------------------------------------------
# Quy hoạch tuyến tính (Linear Programming) — hỗn hợp
# ------------------------------------------------------------------

lp_objective_text = "-3*x - 5*y + 2*z"  # đổi dấu: max 3x+5y-2z -> min -(...)
lp_constraint_texts = [
    "x + 2*y + z = 10",
    "2*x + y <= 8",
    "x >= 0",
    "y >= 0",
    "z >= 0",
]

(lp_objective_expr, lp_constraints, lp_variables,
 lp_objective, lp_constraint_value, lp_constraint_gradient) = build_problem(
    lp_objective_text, lp_constraint_texts
)

lp_init_box = [(0.0, 10.0)] * len(lp_variables)

lp_constraint_value = build_scaled_constraint_value(
    lp_constraints, lp_constraint_value, lp_constraint_gradient, lp_init_box,
)

show_problem(lp_objective_expr, lp_constraints, lp_variables)

lp_ga_runs = [
    genetic_algorithm(
        lp_objective, lp_constraints, lp_constraint_value, lp_init_box,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]
lp_ga_result = functools.reduce(
    lambda x, y: y if is_better(
        y["fun"], y["violation"], x["fun"], x["violation"]
    ) else x,
    lp_ga_runs,
)

lp_scipy_result = scipy_slsqp(
    lp_objective, lp_constraints, lp_constraint_value, lp_init_box,
    n_starts=SLSQP_N_STARTS,
    seed=42,
)

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    lp_ga_result, lp_variables,
)

if N_RUNS > 1:
    show_statistics(lp_ga_runs)

show_solution(
    "SCIPY SLSQP — đa điểm khởi tạo",
    lp_scipy_result, lp_variables,
)

show_comparison(
    [
        ("Genetic Algorithm", lp_ga_result),
        ("SciPy SLSQP", lp_scipy_result),
    ],
    lp_variables,
)


$$
\begin{aligned}
&\underset{x, y, z}{\text{minimize}} \quad && f\left(x, y, z\right) = - 3 x - 5 y + 2 z \\
&\text{subject to} \quad && x + 2 y + z - 10\ =\ 0 \\
& \quad && - 2 x - y + 8\ \ge\ 0 \\
& \quad && x\ \ge\ 0 \\
& \quad && y\ \ge\ 0 \\
& \quad && z\ \ge\ 0 \\
\end{aligned}
$$

**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x &= 1.8299105946 \\ y &= 4.0850452318 \\ z &= -1.0466 \times 10^{-7}\end{aligned}$$
$$f^{*} = -25.9149581523$$

| | |
|---|---|
| Thời gian chạy | $83.377050\ \text{s}$ |
| Thỏa mãn ràng buộc từ thế hệ | $327 / 500$ |

**Thống kê qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = -25.9149581523$ |
| Trung bình | $\bar{f} = -25.8060495325$ |
| Tệ nhất | $\max f = -25.6383604792$ |
| Độ lệch chuẩn | $\sigma = 0.1203271172$ |

**SCIPY SLSQP — đa điểm khởi tạo**

$$\begin{aligned}x &= 2.0000000000 \\ y &= 4.0000000000 \\ z &= -2.3849 \times 10^{-13}\end{aligned}$$
$$f^{*} = -26.0000000000$$

| | |
|---|---|
| Thời gian chạy | $0.416701\ \text{s}$ |
| SLSQP hội tụ | $30/30$ điểm khởi tạo |

| Phương pháp | $f^{*}$ | $x$ | $y$ | $z$ |
|---|---|---|---|---|
| Genetic Algorithm | $-25.9149581523$ | $1.8299105946$ | $4.0850452318$ | $-1.0466 \times 10^{-7}$ |
| SciPy SLSQP | $-26.0000000000$ | $2.0000000000$ | $4.0000000000$ | $-2.3849 \times 10^{-13}$ |

---
## ④ Quy hoạch bình phương (Quadratic Programming — hỗn hợp)

$$\min_{x,y,z} f(x,y,z) = x^2+2y^2+z^2-4x-6y \quad \text{s.t.} \quad
x+y+2z=5,\ \ x+y\le3,\ \ x,y,z\ge0$$

Hàm mục tiêu là **bậc hai lồi** (hệ số $x^2,y^2,z^2$ đều dương, không
có số hạng chéo $xy,yz,xz$), ràng buộc đều tuyến tính — quy hoạch bình
phương (QP) với miền lồi, nên có đúng **một** cực tiểu toàn cục.

$$x^{*}=\tfrac{5}{3}\approx1.666667,\quad y^{*}=\tfrac{4}{3}
\approx1.333333,\quad z^{*}=1, \qquad f^{*}=-\tfrac{22}{3}\approx
-7.333333$$

Ràng buộc $x+y\le3$ **chặt** ($\tfrac{5}{3}+\tfrac{4}{3}=3$ đúng bằng
biên) — tại nghiệm tối ưu, nếu không có ràng buộc này thì $f$ còn giảm
được nữa (cực tiểu không ràng buộc của $f$ nằm ngoài miền khả thi).

In [5]:
# ------------------------------------------------------------------
# Quy hoạch bình phương (Quadratic Programming) — hỗn hợp
# ------------------------------------------------------------------

qp_objective_text = "x^2 + 2*y^2 + z^2 - 4*x - 6*y"
qp_constraint_texts = [
    "x + y + 2*z = 5",
    "x + y <= 3",
    "x >= 0",
    "y >= 0",
    "z >= 0",
]

(qp_objective_expr, qp_constraints, qp_variables,
 qp_objective, qp_constraint_value, qp_constraint_gradient) = build_problem(
    qp_objective_text, qp_constraint_texts
)

qp_init_box = [(0.0, 5.0)] * len(qp_variables)

qp_constraint_value = build_scaled_constraint_value(
    qp_constraints, qp_constraint_value, qp_constraint_gradient, qp_init_box,
)

show_problem(qp_objective_expr, qp_constraints, qp_variables)

qp_ga_runs = [
    genetic_algorithm(
        qp_objective, qp_constraints, qp_constraint_value, qp_init_box,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]
qp_ga_result = functools.reduce(
    lambda x, y: y if is_better(
        y["fun"], y["violation"], x["fun"], x["violation"]
    ) else x,
    qp_ga_runs,
)

qp_scipy_result = scipy_slsqp(
    qp_objective, qp_constraints, qp_constraint_value, qp_init_box,
    n_starts=SLSQP_N_STARTS,
    seed=42,
)

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    qp_ga_result, qp_variables,
)

if N_RUNS > 1:
    show_statistics(qp_ga_runs)

show_solution(
    "SCIPY SLSQP — đa điểm khởi tạo",
    qp_scipy_result, qp_variables,
)

show_comparison(
    [
        ("Genetic Algorithm", qp_ga_result),
        ("SciPy SLSQP", qp_scipy_result),
    ],
    qp_variables,
)


$$
\begin{aligned}
&\underset{x, y, z}{\text{minimize}} \quad && f\left(x, y, z\right) = x^{2} - 4 x + 2 y^{2} - 6 y + z^{2} \\
&\text{subject to} \quad && x + y + 2 z - 5\ =\ 0 \\
& \quad && - x - y + 3\ \ge\ 0 \\
& \quad && x\ \ge\ 0 \\
& \quad && y\ \ge\ 0 \\
& \quad && z\ \ge\ 0 \\
\end{aligned}
$$

**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x &= 1.6773442628 \\ y &= 1.3226557415 \\ z &= 0.9999987769\end{aligned}$$
$$f^{*} = -7.3329937494$$

| | |
|---|---|
| Thời gian chạy | $102.002758\ \text{s}$ |
| Thỏa mãn ràng buộc từ thế hệ | $356 / 500$ |

**Thống kê qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = -7.3329937494$ |
| Trung bình | $\bar{f} = -7.3297805594$ |
| Tệ nhất | $\max f = -7.3239910924$ |
| Độ lệch chuẩn | $\sigma = 0.0041020207$ |

**SCIPY SLSQP — đa điểm khởi tạo**

$$\begin{aligned}x &= 1.6666666747 \\ y &= 1.3333333253 \\ z &= 1.0000000000\end{aligned}$$
$$f^{*} = -7.3333333333$$

| | |
|---|---|
| Thời gian chạy | $0.256324\ \text{s}$ |
| SLSQP hội tụ | $30/30$ điểm khởi tạo |

| Phương pháp | $f^{*}$ | $x$ | $y$ | $z$ |
|---|---|---|---|---|
| Genetic Algorithm | $-7.3329937494$ | $1.6773442628$ | $1.3226557415$ | $0.9999987769$ |
| SciPy SLSQP | $-7.3333333333$ | $1.6666666747$ | $1.3333333253$ | $1.0000000000$ |

---
## ⑤ Tối ưu lồi phi tuyến (Convex Optimization — hàm mũ)

$$\min_{x,y} f(x,y) = e^{-x}+e^{-2y} \quad \text{s.t.} \quad
2x+3y=6,\ \ x^2+y^2\le4,\ \ x,y\ge0$$

$e^{-x}$ và $e^{-2y}$ đều là hàm **lồi**, tổng của chúng cũng lồi;
miền ràng buộc (giao của một đường thẳng, một hình tròn, và góc phần
tư dương) là **tập lồi** — nên bài toán có đúng một cực tiểu toàn cục.

$$x^{*}\approx1.590993,\quad y^{*}\approx0.939338, \qquad
f^{*}\approx0.356515$$

Khác hai bài trên: ràng buộc hình tròn $x^2+y^2\le4$ **không chặt**
tại nghiệm ($x^{*2}+y^{*2}\approx3.41<4$) — chỉ đường thẳng $2x+3y=6$
và điều kiện không âm mới thực sự chi phối nghiệm; hình tròn ở đây chỉ
đóng vai trò giới hạn miền tìm kiếm, không ảnh hưởng nghiệm tối ưu.

In [6]:
# ------------------------------------------------------------------
# Tối ưu lồi phi tuyến (hàm mũ)
# ------------------------------------------------------------------

convex_objective_text = "exp(-x) + exp(-2*y)"
convex_constraint_texts = [
    "2*x + 3*y = 6",
    "x^2 + y^2 <= 4",
    "x >= 0",
    "y >= 0",
]

(convex_objective_expr, convex_constraints, convex_variables,
 convex_objective, convex_constraint_value,
 convex_constraint_gradient) = build_problem(
    convex_objective_text, convex_constraint_texts
)

convex_init_box = [(0.0, 2.0)] * len(convex_variables)

convex_constraint_value = build_scaled_constraint_value(
    convex_constraints, convex_constraint_value,
    convex_constraint_gradient, convex_init_box,
)

show_problem(convex_objective_expr, convex_constraints, convex_variables)

convex_ga_runs = [
    genetic_algorithm(
        convex_objective, convex_constraints, convex_constraint_value,
        convex_init_box,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]
convex_ga_result = functools.reduce(
    lambda x, y: y if is_better(
        y["fun"], y["violation"], x["fun"], x["violation"]
    ) else x,
    convex_ga_runs,
)

convex_scipy_result = scipy_slsqp(
    convex_objective, convex_constraints, convex_constraint_value,
    convex_init_box,
    n_starts=SLSQP_N_STARTS,
    seed=42,
)

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    convex_ga_result, convex_variables,
)

if N_RUNS > 1:
    show_statistics(convex_ga_runs)

show_solution(
    "SCIPY SLSQP — đa điểm khởi tạo",
    convex_scipy_result, convex_variables,
)

show_comparison(
    [
        ("Genetic Algorithm", convex_ga_result),
        ("SciPy SLSQP", convex_scipy_result),
    ],
    convex_variables,
)


$$
\begin{aligned}
&\underset{x, y}{\text{minimize}} \quad && f\left(x, y\right) = e^{- 2 y} + e^{- x} \\
&\text{subject to} \quad && 2 x + 3 y - 6\ =\ 0 \\
& \quad && - x^{2} - y^{2} + 4\ \ge\ 0 \\
& \quad && x\ \ge\ 0 \\
& \quad && y\ \ge\ 0 \\
\end{aligned}
$$

**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x &= 1.5723680701 \\ y &= 0.9517554079\end{aligned}$$
$$f^{*} = 0.3565975317$$

| | |
|---|---|
| Thời gian chạy | $100.972967\ \text{s}$ |
| Thỏa mãn ràng buộc từ thế hệ | $203 / 500$ |

**Thống kê qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = 0.3565975317$ |
| Trung bình | $\bar{f} = 0.3567642194$ |
| Tệ nhất | $\max f = 0.3569453855$ |
| Độ lệch chuẩn | $\sigma = 0.0001423793$ |

**SCIPY SLSQP — đa điểm khởi tạo**

$$\begin{aligned}x &= 1.5909934070 \\ y &= 0.9393377287\end{aligned}$$
$$f^{*} = 0.3565154830$$

| | |
|---|---|
| Thời gian chạy | $0.483346\ \text{s}$ |
| SLSQP hội tụ | $30/30$ điểm khởi tạo |

| Phương pháp | $f^{*}$ | $x$ | $y$ |
|---|---|---|---|
| Genetic Algorithm | $0.3565975317$ | $1.5723680701$ | $0.9517554079$ |
| SciPy SLSQP | $0.3565154830$ | $1.5909934070$ | $0.9393377287$ |